In [27]:
import pandas as pd
import requests
import time
import os
from dotenv import load_dotenv
from urllib.parse import quote

class NaverSearchAPI:
    def __init__(self):
        # .env 파일에서 환경변수 로드
        load_dotenv()
        
        self.client_id = os.getenv('Client_ID')
        self.client_secret = os.getenv('Client_Secret')
        
        # API 키 확인
        if not self.client_id or not self.client_secret:
            raise ValueError("API 키가 설정되지 않았습니다. .env 파일을 확인하세요.")
    
    def get_search_count(self, keyword, search_type="blog"):
        """네이버 검색 API로 검색 결과 수 가져오기"""
        
        # API URL 설정
        api_urls = {
            "blog": "https://openapi.naver.com/v1/search/blog.json",
            "news": "https://openapi.naver.com/v1/search/news.json",
            "cafe": "https://openapi.naver.com/v1/search/cafearticle.json",
            "web": "https://openapi.naver.com/v1/search/webkr.json"
        }
        
        url = api_urls.get(search_type, api_urls["blog"])
        
        # 헤더 설정
        headers = {
            'X-Naver-Client-Id': self.client_id,
            'X-Naver-Client-Secret': self.client_secret
        }
        
        # 파라미터 설정
        params = {
            'query': keyword,
            'display': 1,  # 1개만 가져와서 total만 확인
            'start': 1
        }
        
        try:
            response = requests.get(url, headers=headers, params=params)
            
            if response.status_code == 200:
                data = response.json()
                return data.get('total', 0)
            else:
                print(f"API 오류 ({search_type}): {response.status_code} - {keyword}")
                return 0
                
        except Exception as e:
            print(f"API 호출 오류: {e} - {keyword}")
            return 0
    
    def test_api_connection(self):
        """API 연결 테스트"""
        test_keyword = "김치찌개"
        test_result = self.get_search_count(test_keyword)
        
        if test_result > 0:
            print(f"API 연결 테스트 성공: '{test_keyword}' 블로그 검색 결과 {test_result:,}개")
            return True
        else:
            print("API 연결 테스트 실패. .env 파일의 API 키를 확인하세요.")
            return False
    
    def collect_search_counts(self, keywords):
        """모든 키워드의 검색 결과 수 수집"""
        
        results = []
        total_keywords = len(keywords)
        
        print(f"총 {total_keywords}개 키워드 검색 시작")
        print("-" * 50)
        
        for i, keyword in enumerate(keywords, 1):
            print(f"[{i}/{total_keywords}] {keyword}")
            
            # 각 검색 타입별로 결과 수 가져오기
            blog_count = self.get_search_count(keyword, "blog")
            time.sleep(0.1)
            
            news_count = self.get_search_count(keyword, "news")
            time.sleep(0.1)
            
            cafe_count = self.get_search_count(keyword, "cafe")
            time.sleep(0.1)
            
            web_count = self.get_search_count(keyword, "web")
            time.sleep(0.1)
            
            # 결과 저장
            total_count = blog_count + news_count + cafe_count + web_count
            
            results.append({
                '키워드': keyword,
                '블로그_검색수': blog_count,
                '뉴스_검색수': news_count,
                '카페_검색수': cafe_count,
                '웹_검색수': web_count,
                '총합': total_count
            })
            
            print(f"  블로그: {blog_count:,}, 뉴스: {news_count:,}, 카페: {cafe_count:,}, 웹: {web_count:,}")
            print(f"  총합: {total_count:,}")
            
            # API 호출 제한을 위한 대기
            time.sleep(0.5)
        
        return pd.DataFrame(results)

def load_keywords_from_csv():
    """CSV 파일에서 키워드 읽기"""
    try:
        print("CSV 파일 읽기 중...")
        df = pd.read_csv('식당대12중53소132상세메뉴379분류.csv')
        
        keywords = []
        for menu in df['상세메뉴'].dropna():
            for item in str(menu).split(','):
                item = item.strip()
                if item:
                    keywords.append(item)
        
        print(f"총 {len(keywords)}개 키워드 로딩 완료")
        return keywords
        
    except FileNotFoundError:
        print("오류: '식당대12중53소132상세메뉴379분류.csv' 파일을 찾을 수 없습니다.")
        print("현재 폴더에 CSV 파일이 있는지 확인하세요.")
        return None
    except Exception as e:
        print(f"CSV 파일 읽기 오류: {e}")
        return None

def process_all_keywords(keywords):
    """전체 키워드 처리"""
    print(f"\n전체 {len(keywords)}개 키워드를 처리합니다.")
    return keywords

def save_results_to_excel(results_df):
    """결과를 엑셀 파일로 저장"""
    # 결과 정렬 (총합 기준 내림차순)
    results_df = results_df.sort_values('총합', ascending=False)
    
    # 엑셀 파일로 저장
    output_filename = "네이버_검색_절대숫자.xlsx"
    results_df.to_excel(output_filename, index=False)
    
    print("\n" + "=" * 50)
    print("수집 완료!")
    print(f"파일 저장: {output_filename}")
    print(f"총 처리된 키워드: {len(results_df)}개")
    
    # 상위 10개 키워드 출력
    print("\n상위 10개 키워드:")
    print("-" * 50)
    top_10 = results_df.head(10)
    for _, row in top_10.iterrows():
        print(f"{row['키워드']}: {row['총합']:,}개")
    
    # 통계 정보
    print(f"\n통계 정보:")
    print(f"평균 검색 결과 수: {results_df['총합'].mean():,.0f}개")
    print(f"최대 검색 결과 수: {results_df['총합'].max():,}개")
    print(f"최소 검색 결과 수: {results_df['총합'].min():,}개")
    
    return results_df

def create_env_file_template():
    """환경변수 파일 템플릿 생성"""
    env_template = """# 네이버 개발자센터에서 발급받은 API 키를 입력하세요
# https://developers.naver.com/apps/#/register

Client_ID=your_client_id_here
Client_Secret=your_client_secret_here
"""
    
    with open('.env', 'w', encoding='utf-8') as f:
        f.write(env_template)
    
    print(".env 파일 템플릿을 생성했습니다.")
    print("파일을 열어서 API 키를 입력한 후 다시 실행하세요.")

def main():
    print("네이버 검색 API를 사용한 전체 키워드 검색량 조사")
    print("=" * 60)
    
    # .env 파일 확인
    if not os.path.exists('.env'):
        print("오류: .env 파일이 없습니다.")
        print("현재 폴더에 .env 파일이 있는지 확인하세요.")
        return
    
    # API 클래스 초기화
    try:
        api = NaverSearchAPI()
    except ValueError as e:
        print(f"오류: {e}")
        print("\n.env 파일을 확인하세요:")
        print("Client_ID=your_client_id")
        print("Client_Secret=your_client_secret")
        return
    
    # API 연결 테스트
    print("\nAPI 연결 테스트 중...")
    if not api.test_api_connection():
        return
    
    # 키워드 로드
    keywords = load_keywords_from_csv()
    if keywords is None:
        return
    
    # 전체 키워드 처리
    selected_keywords = process_all_keywords(keywords)
    
    # 예상 소요 시간 계산
    estimated_time = len(selected_keywords) * 0.5 / 60  # 키워드당 0.5초 * 분 변환
    print(f"예상 소요 시간: 약 {estimated_time:.1f}분")
    
    # 처리 시작 확인
    start_confirm = input("\n처리를 시작하시겠습니까? (y/n): ").strip().lower()
    if start_confirm != 'y':
        print("처리를 중단합니다.")
        return
    
    # 검색 결과 수 수집
    try:
        print(f"\n전체 키워드 검색량 조사 시작...")
        results_df = api.collect_search_counts(selected_keywords)
        
        # 결과 저장 및 출력
        save_results_to_excel(results_df)
        
    except Exception as e:
        print(f"처리 중 오류 발생: {e}")
        return

if __name__ == "__main__":
    main()

네이버 검색 API를 사용한 전체 키워드 검색량 조사

API 연결 테스트 중...
API 연결 테스트 성공: '김치찌개' 블로그 검색 결과 3,836,003개
CSV 파일 읽기 중...
총 381개 키워드 로딩 완료

전체 381개 키워드를 처리합니다.
예상 소요 시간: 약 3.2분

처리를 시작하시겠습니까? (y/n): y

전체 키워드 검색량 조사 시작...
총 381개 키워드 검색 시작
--------------------------------------------------
[1/381] 제육볶음
  블로그: 1,787,231, 뉴스: 30,150, 카페: 392,437, 웹: 2,610,834
  총합: 4,820,652
[2/381] 매운제육볶음
  블로그: 202,505, 뉴스: 2,090, 카페: 26,303, 웹: 674,817
  총합: 905,715
[3/381] 두부제육볶음
  블로그: 283,475, 뉴스: 2,433, 카페: 63,762, 웹: 883,520
  총합: 1,233,190
[4/381] 된장찌개
  블로그: 6,118,976, 뉴스: 65,879, 카페: 652,046, 웹: 4,481,974
  총합: 11,318,875
[5/381] 김치찌개
  블로그: 3,835,999, 뉴스: 96,158, 카페: 821,924, 웹: 8,037,007
  총합: 12,791,088
[6/381] 청국장찌개
  블로그: 496,202, 뉴스: 6,397, 카페: 79,886, 웹: 1,261,677
  총합: 1,844,162
[7/381] 콩나물무침
  블로그: 1,929,512, 뉴스: 7,745, 카페: 316,020, 웹: 2,214,245
  총합: 4,467,522
[8/381] 시금치나물
  블로그: 467,914, 뉴스: 10,147, 카페: 198,423, 웹: 1,204,794
  총합: 1,881,278
[9/381] 도라지무침
  블로그: 285,641, 뉴스: 2,942, 카페: 90,127, 웹: 846

[96/381] 멍게무침
  블로그: 226,200, 뉴스: 2,335, 카페: 14,326, 웹: 361,008
  총합: 603,869
[97/381] 곱창
  블로그: 4,716,216, 뉴스: 67,949, 카페: 1,274,698, 웹: 6,208,994
  총합: 12,267,857
[98/381] 막창
  블로그: 2,126,358, 뉴스: 20,352, 카페: 427,652, 웹: 2,810,909
  총합: 5,385,271
[99/381] 순대
  블로그: 3,648,080, 뉴스: 85,551, 카페: 1,172,274, 웹: 7,429,063
  총합: 12,334,968
[100/381] 마른오징어
  블로그: 225,542, 뉴스: 10,818, 카페: 84,008, 웹: 1,925,290
  총합: 2,245,658
[101/381] 육포
  블로그: 496,745, 뉴스: 32,984, 카페: 229,216, 웹: 1,766,672
  총합: 2,525,617
[102/381] 땅콩
  블로그: 4,106,712, 뉴스: 164,578, 카페: 1,201,614, 웹: 7,890,298
  총합: 13,363,202
[103/381] 신라면
  블로그: 1,340,026, 뉴스: 53,594, 카페: 274,410, 웹: 11,671,660
  총합: 13,339,690
[104/381] 너구리
  블로그: 750,984, 뉴스: 56,850, 카페: 504,133, 웹: 4,498,157
  총합: 5,810,124
[105/381] 짜파게티
  블로그: 756,016, 뉴스: 19,318, 카페: 280,951, 웹: 1,990,644
  총합: 3,046,929
[106/381] 컵라면
  블로그: 2,120,906, 뉴스: 92,864, 카페: 759,194, 웹: 4,161,342
  총합: 7,134,306
[107/381] 용기면
  블로그: 11,538, 뉴스: 20,667, 카페: 3,585, 웹: 16,586,76

[199/381] 카레우동
  블로그: 624,789, 뉴스: 4,169, 카페: 59,532, 웹: 1,119,813
  총합: 1,808,303
[200/381] 자루소바
  블로그: 61,087, 뉴스: 516, 카페: 3,905, 웹: 290,516
  총합: 356,024
[201/381] 온소바
  블로그: 594,137, 뉴스: 4,739, 카페: 28,959, 웹: 1,204,992
  총합: 1,832,827
[202/381] 메밀소바
  블로그: 440,844, 뉴스: 6,735, 카페: 43,930, 웹: 902,435
  총합: 1,393,944
[203/381] 돈까스
  블로그: 7,121,590, 뉴스: 43,915, 카페: 1,775,592, 웹: 9,133,780
  총합: 18,074,877
[204/381] 치킨가츠
  블로그: 197,056, 뉴스: 228, 카페: 49,652, 웹: 4,234,225
  총합: 4,481,161
[205/381] 생선까스
  블로그: 294,753, 뉴스: 1,394, 카페: 74,680, 웹: 768,338
  총합: 1,139,165
[206/381] 새우텐푸라
  블로그: 45,178, 뉴스: 29, 카페: 2,314, 웹: 47,112
  총합: 94,633
[207/381] 야채텐푸라
  블로그: 10,809, 뉴스: 17, 카페: 769, 웹: 16,201
  총합: 27,796
[208/381] 모둠텐푸라
  블로그: 1,017, 뉴스: 5, 카페: 33, 웹: 2,073
  총합: 3,128
[209/381] 가라아게
  블로그: 749,909, 뉴스: 4,964, 카페: 35,637, 웹: 881,786
  총합: 1,672,296
[210/381] 치킨가라아게
  블로그: 357,838, 뉴스: 3,273, 카페: 15,965, 웹: 748,990
  총합: 1,126,066
[211/381] 닭튀김
  블로그: 319,251, 뉴스: 4,979, 카페: 35,965, 웹

[302/381] 간장치킨
  블로그: 1,440,795, 뉴스: 27,422, 카페: 178,702, 웹: 3,045,763
  총합: 4,692,682
[303/381] 마늘치킨
  블로그: 1,325,915, 뉴스: 26,760, 카페: 124,566, 웹: 2,549,542
  총합: 4,026,783
[304/381] 허니머스타드
  블로그: 136,185, 뉴스: 2,949, 카페: 22,155, 웹: 337,164
  총합: 498,453
[305/381] 뿌링클
  블로그: 344,671, 뉴스: 18,143, 카페: 118,392, 웹: 710,141
  총합: 1,191,347
[306/381] 치즈가루
  블로그: 2,021,979, 뉴스: 22,430, 카페: 213,029, 웹: 3,120,740
  총합: 5,378,178
[307/381] 어니언
  블로그: 826,097, 뉴스: 19,539, 카페: 89,301, 웹: 1,223,399
  총합: 2,158,336
[308/381] 순살치킨
  블로그: 1,133,110, 뉴스: 31,760, 카페: 169,076, 웹: 1,717,071
  총합: 3,051,017
[309/381] 팝콘치킨
  블로그: 248,798, 뉴스: 7,438, 카페: 35,729, 웹: 973,489
  총합: 1,265,454
[310/381] 치킨텐더
  블로그: 330,207, 뉴스: 8,908, 카페: 54,357, 웹: 821,118
  총합: 1,214,590
[311/381] 윙
  블로그: 1,402,469, 뉴스: 132,588, 카페: 1,364,042, 웹: 7,810,290
  총합: 10,709,389
[312/381] 다리
  블로그: 27,291,648, 뉴스: 1,841,258, 카페: 12,585,285, 웹: 51,996,568
  총합: 93,714,759
[313/381] 가슴살
  블로그: 3,823,213, 뉴스: 107,645, 카페: 1,328,213, 웹: